# Average in-air delay for each aircraft manufacturer

In [1]:
import org.apache.spark

Intitializing Scala interpreter ...

Spark Web UI available at http://AndreasPC.station:4041
SparkContext available as 'sc' (version = 3.5.1, master = local[*], app id = local-1739274915620)
SparkSession available as 'spark'


import org.apache.spark


In [ ]:
// DO NOT EXECUTE - this is needed just to avoid showing errors in the following cells
val sc = spark.SparkContext.getOrCreate()

## Parsing the data

Firstly, we need some functions that can parse our CSV files.

In [ ]:
def getInt(str:String) : Int = {
    if (str.forall(Character.isDigit))
        str.toInt
    else
        -1
}

def parse_flight(line: String) = {
    val parts = line.split(",")
    val year = getInt(parts(0))
    val month = getInt(parts(1))
    val day = getInt(parts(2))
    val dep_time = parts(4)
    val dep_delay = getInt(parts(15))
    val arr_time = parts(6)
    val arr_delay = getInt(parts(14))
    val carrier = parts(8)
    val flightnum = parts(9)
    val tailnum = parts(10)
    val origin = parts(16)
    val dest = parts(17)
    (year, month, day, dep_time, dep_delay, arr_time, arr_delay, carrier, tailnum, flightnum, origin, dest)
}


def parse_aircraft(line: String) = {
    val parts = line.split(",")
    if (parts.length != 9) {
        (parts(0), "", "", "", "", 0)
    } else {
    val tailnum = parts(0)
    val manufacturer = parts(2)
    val model = parts(4)
    val type_aircraft = parts(6)
    val engine_type = parts(7)
    val year = getInt(parts(8))
    (tailnum, manufacturer, model, type_aircraft, engine_type, year)
    }
}

getInt: (str: String)Int
parse_flight: (line: String)(Int, Int, Int, String, Int, String, Int, String, String, String, String, String)
parse_aircraft: (line: String)(String, String, String, String, String, Int)


## Loading the data

Then we can load our CSV files as RDD's. Then we map with the index to get rid of the CSV file headers and parse flight and aircraft records.

In [8]:
val rddFlights = sc.textFile("./../../../../datasets/project/flights.csv").mapPartitionsWithIndex { (idx, iter) => if (idx == 0) iter.drop(1) else iter }.map(parse_flight)
val rddAircraft = sc.textFile("./../../../../datasets/project/plane-data.csv").mapPartitionsWithIndex { (idx, iter) => if (idx == 0) iter.drop(1) else iter }.map(parse_aircraft)

rddFlights: org.apache.spark.rdd.RDD[(Int, Int, Int, String, Int, String, Int, String, String, String, String, String)] = MapPartitionsRDD[9] at map at <console>:27
rddAircraft: org.apache.spark.rdd.RDD[(String, String, String, String, String, Int)] = MapPartitionsRDD[13] at map at <console>:28


Some of the aircraft records are incomplete in the dataset. We handle this by filling with empty values in the `parse_aircraft` function. We want to filter out the incomplete records from the RDD, so we can filter based on the registration year, which is set to 0 for incomplete records. 

In [ ]:
val rddAircraftFiltered = rddAircraft.filter(x => x._6 > 0)

We now want to decide which pattern our job shall have. The flight dataset is very large, so a join-and-aggregate pattern would probably use a lot of resources. We opt for aggregation before joining. Since we want to find out the delay of each aircraft manufacturer, we can transform each flight record to only include information about the aircraft and the in-air delay. In our case, that will be the tail number and (arrival delay - departure delay). Then we can aggregate total delay and amount of flights on each tail number.

In [ ]:
val rddTailNumsDelays = rddFlights
    .map(x => (x._9, x._7 - x._5))
    .aggregateByKey((0.0, 0.0))((acc, value) => (acc._1 + 1, acc._2 + value), (acc1, acc2) => (acc1._1 + acc2._1, acc1._2 + acc2._2))